In [ ]:
from aereo.builtins.search import search_earthaccess
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.cache import TaskResultCache
from datetime import datetime, timezone
from shapely.geometry import Polygon
from aereo.executors import LocalExecutor

from aereo.pipeline import ExtractionJob

# AOI polygon — Chocón reservoir, Argentina (inlined so the notebook has no file dependencies).
aoi_polygon = Polygon(
    [
        (-68.90986824592407, -39.23705421799603),
        (-68.65925870907353, -39.23705421799603),
        (-68.65925870907353, -39.41589522092947),
        (-68.90986824592407, -39.41589522092947),
        (-68.90986824592407, -39.23705421799603),
    ]
)

assets = search_earthaccess(
    collections={"VJ202IMG": ["I04"], "VJ203IMG": []},
    intersects=aoi_polygon,
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(
        2024, 1, 1, hour=23, minute=59, second=59, tzinfo=timezone.utc
    ),
)

In [ ]:
# Load the job from the Hydra config package.
job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_viirs",
)

tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=5,
)

In [ ]:
# Extract
print("Extracting...")
artifacts = job.execute(
    tasks,
    executor=LocalExecutor(workers=1, cache=TaskResultCache()),
)
print(f"✓ Extracted {len(artifacts)} artifacts")

In [ ]:
import rioxarray

rioxarray.open_rasterio(artifacts.iloc[0].uri).plot()

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, ds_factor=10, cmap="inferno")